# MotoGP Data Download

This notebook downloads the MotoGP CSV datasets to the local `/data/raw` directory.

In [ ]:
import requests
import os
from pathlib import Path
import time

## Setup Data Directory

In [ ]:
data_dir = Path("data/raw")
data_dir.mkdir(parents=True, exist_ok=True)
print(f"Created directory: {data_dir}")

## Dataset Configuration

In [ ]:
datasets = {
    "constructure_world_championship": "https://raw.githubusercontent.com/dipedilans/assets/refs/heads/main/Moto_GP_World_Championship_1949-2022/constructure-world-championship.csv",
    "grand_prix_events_held": "https://raw.githubusercontent.com/dipedilans/assets/refs/heads/main/Moto_GP_World_Championship_1949-2022/grand-prix-events-held.csv",
    "grand_prix_race_winners": "https://raw.githubusercontent.com/dipedilans/assets/refs/heads/main/Moto_GP_World_Championship_1949-2022/grand-prix-race-winners.csv",
    "riders_finishing_positions": "https://raw.githubusercontent.com/dipedilans/assets/refs/heads/main/Moto_GP_World_Championship_1949-2022/riders-finishing-positions.csv",
    "riders_info": "https://raw.githubusercontent.com/dipedilans/assets/refs/heads/main/Moto_GP_World_Championship_1949-2022/riders-info.csv",
    "same_nation_podium_lockouts": "https://raw.githubusercontent.com/dipedilans/assets/refs/heads/main/Moto_GP_World_Championship_1949-2022/same-nation-podium-lockouts.csv"
}

print(f"Configured {len(datasets)} datasets for download")

## Download Function

In [ ]:
def download_csv(name, url, data_dir):
    """
    Download a CSV file from URL to the specified directory.
    
    Args:
        name: Name of the dataset
        url: URL to download from
        data_dir: Directory to save the file
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        print(f"Downloading {name}...")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        file_path = data_dir / f"{name}.csv"
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        file_size = file_path.stat().st_size
        print(f"✓ Downloaded {name}.csv ({file_size:,} bytes)")
        return True
        
    except requests.exceptions.RequestException as e:
        print(f"✗ Failed to download {name}: {e}")
        return False
    except Exception as e:
        print(f"✗ Error saving {name}: {e}")
        return False

## Download All Datasets

In [ ]:
print("Starting download of all MotoGP datasets...\n")

successful_downloads = []
failed_downloads = []

for name, url in datasets.items():
    if download_csv(name, url, data_dir):
        successful_downloads.append(name)
    else:
        failed_downloads.append(name)
    
    # Small delay between downloads
    time.sleep(0.5)

print(f"\n=== Download Summary ===")
print(f"Successful: {len(successful_downloads)}/{len(datasets)}")
print(f"Failed: {len(failed_downloads)}/{len(datasets)}")

if failed_downloads:
    print(f"\nFailed downloads: {', '.join(failed_downloads)}")
else:
    print(f"\n🎉 All datasets downloaded successfully to {data_dir}/")

## Verification

In [ ]:
print("\n=== File Verification ===")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in {data_dir}/")

for file_path in sorted(csv_files):
    size = file_path.stat().st_size
    print(f"  {file_path.name}: {size:,} bytes")

if len(csv_files) == len(datasets):
    print(f"\n✓ All {len(datasets)} datasets are present")
else:
    print(f"\n⚠ Expected {len(datasets)} files, found {len(csv_files)}")